<a href="https://colab.research.google.com/github/SajidAhmed-here/apriori-dashboard/blob/main/488dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Comprehensive Dashboard for Frequent Pattern Mining - Ngrok Ready
!pip install plotly dash dash-bootstrap-components pandas mlxtend openpyxl pyngrok -q

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
import dash
from dash import dcc, html, Input, Output, State, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go
import math
from pyngrok import ngrok

# ---------------------------
# Load and parse Excel
# ---------------------------
print("Loading data...")
excel_path = '/content/transactions.xlsx'
df = pd.read_excel(excel_path, header=None)
df.dropna(how='all', inplace=True)

# Detect which column contains items
if df.shape[1] == 1:
    item_col = 0
else:
    item_col = 1

transactions = []
for idx, row in df.iterrows():
    cell = str(row[item_col]).strip()
    if not cell or cell.lower() == 'nan':
        continue
    items = [it.strip() for it in cell.split() if it.strip()]
    transactions.append(items)

n_trans = len(transactions)
print(f"✅ Loaded {n_trans} transactions")

# Transaction encoder
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
items_list = list(te.columns_)

# Helper function for exact counts
def itemset_count(itemset, transactions):
    s = set(itemset)
    return sum(1 for t in transactions if s.issubset(t))

# Color palettes for different visualizations
COLOR_PALETTES = {
    'items': ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9', '#F8C471', '#82E0AA', '#F1948A', '#85C1E9', '#D7BDE2'],
    'itemsets': ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'],
    'support': ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52'],
    'length': ['#FF9999', '#66B2FF', '#99FF99', '#FFB366', '#FF99FF', '#FFFF99', '#99FFFF', '#FFCC99', '#CC99FF', '#99FFCC']
}

# Initialize Dash app with external styles and scripts for better ngrok compatibility
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    meta_tags=[
        {"name": "viewport", "content": "width=device-width, initial-scale=1"}
    ]
)
app.title = "Frequent Pattern Mining Dashboard"

# Disable Dash hot-reload for better ngrok performance
app.config.suppress_callback_exceptions = True

# Layout - Optimized for web hosting
app.layout = dbc.Container([
    # Header
    dbc.Row([
        dbc.Col([
            html.H1("🎯 Frequent Pattern Mining Dashboard",
                   className="text-center mb-4",
                   style={'color': '#2c3e50', 'fontWeight': 'bold', 'marginTop': '20px'}),
            html.P("Interactive analysis of frequent itemsets using Apriori algorithm",
                  className="text-center text-muted mb-4")
        ])
    ]),

    # Control Panel
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader("⚙️ Mining Controls", className="h5", style={'backgroundColor': '#f8f9fa'}),
                dbc.CardBody([
                    dbc.Row([
                        dbc.Col([
                            html.Label("Minimum Support:", className="fw-bold mb-3"),
                            dcc.Slider(
                                id='support-slider',
                                min=0.01, max=0.5, step=0.01, value=0.1,
                                marks={i/10: f'{i/10:.1f}' for i in range(1, 6)},
                                tooltip={'placement': 'bottom', 'always_visible': True},
                                updatemode='drag'
                            ),
                            html.Div(id='support-value', className="text-center text-primary fw-bold mt-2")
                        ], width=6),
                        dbc.Col([
                            html.Label("Itemset Length Filter:", className="fw-bold mb-3"),
                            dcc.Dropdown(
                                id='length-filter',
                                options=[{'label': f'All Itemsets', 'value': None}] +
                                        [{'label': f'{i}-Itemsets', 'value': i} for i in range(1, 6)],
                                value=None,
                                placeholder="Select itemset length..."
                            ),
                            html.Br(),
                            html.Label("Sort By:", className="fw-bold mb-3"),
                            dcc.Dropdown(
                                id='sort-by',
                                options=[
                                    {'label': 'Support (High to Low)', 'value': 'support'},
                                    {'label': 'Count (High to Low)', 'value': 'count'},
                                    {'label': 'Length', 'value': 'length'}
                                ],
                                value='support'
                            )
                        ], width=6)
                    ]),
                    dbc.Row([
                        dbc.Col([
                            dbc.Button("🚀 Run Apriori Algorithm",
                                      id='run-button',
                                      color="primary",
                                      size="lg",
                                      className="w-100 mt-3",
                                      n_clicks=0)
                        ])
                    ])
                ])
            ], className="mb-4 shadow-sm")
        ])
    ]),

    # Statistics Cards
    dbc.Row([
        dbc.Col([dbc.Card([
            dbc.CardBody([
                html.H4("📊 Transactions", className="card-title", style={'fontSize': '1.1rem'}),
                html.H2(id="total-transactions", className="text-primary", style={'margin': '0'})
            ])
        ], className="shadow-sm")], width=3),
        dbc.Col([dbc.Card([
            dbc.CardBody([
                html.H4("🏷️ Unique Items", className="card-title", style={'fontSize': '1.1rem'}),
                html.H2(id="unique-items", className="text-success", style={'margin': '0'})
            ])
        ], className="shadow-sm")], width=3),
        dbc.Col([dbc.Card([
            dbc.CardBody([
                html.H4("📈 Min Support", className="card-title", style={'fontSize': '1.1rem'}),
                html.H2(id="min-support", className="text-warning", style={'margin': '0'})
            ])
        ], className="shadow-sm")], width=3),
        dbc.Col([dbc.Card([
            dbc.CardBody([
                html.H4("🔍 Itemsets Found", className="card-title", style={'fontSize': '1.1rem'}),
                html.H2(id="itemsets-found", className="text-info", style={'margin': '0'})
            ])
        ], className="shadow-sm")], width=3)
    ], className="mb-4"),

    # Results Tabs
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dbc.Tabs([
                        # Tab 1: Frequent Itemsets Table
                        dbc.Tab([
                            html.Div([
                                html.Div(id="results-info", className="mb-3"),
                                dash_table.DataTable(
                                    id='results-table',
                                    page_size=15,
                                    page_current=0,
                                    style_cell={
                                        'textAlign': 'left',
                                        'padding': '12px',
                                        'fontFamily': 'Arial, sans-serif',
                                        'fontSize': '14px',
                                        'border': '1px solid #dee2e6'
                                    },
                                    style_header={
                                        'backgroundColor': '#2c3e50',
                                        'color': 'white',
                                        'fontWeight': 'bold',
                                        'fontSize': '14px',
                                        'textAlign': 'center'
                                    },
                                    style_data={
                                        'backgroundColor': '#ffffff',
                                        'color': '#2c3e50'
                                    },
                                    style_data_conditional=[
                                        {
                                            'if': {'row_index': 'odd'},
                                            'backgroundColor': '#f8f9fa'
                                        }
                                    ],
                                    style_table={
                                        'overflowX': 'auto',
                                        'border': '1px solid #dee2e6',
                                        'borderRadius': '5px'
                                    }
                                )
                            ], className="mt-3")
                        ], label="📋 Frequent Itemsets", tab_style={"margin": "5px"}, tabClassName="fw-bold"),

                        # Tab 2: Visualizations
                        dbc.Tab([
                            dbc.Row([
                                dbc.Col([dcc.Graph(id='support-distribution')], width=6, className="mb-4"),
                                dbc.Col([dcc.Graph(id='itemset-length-dist')], width=6, className="mb-4")
                            ]),
                            dbc.Row([
                                dbc.Col([dcc.Graph(id='top-itemsets-chart')], width=12, className="mb-4")
                            ])
                        ], label="📊 Visualizations", tab_style={"margin": "5px"}, tabClassName="fw-bold"),

                        # Tab 3: Transaction Analysis
                        dbc.Tab([
                            dbc.Row([
                                dbc.Col([dcc.Graph(id='transaction-length-dist')], width=6, className="mb-4"),
                                dbc.Col([dcc.Graph(id='top-items-chart')], width=6, className="mb-4")
                            ])
                        ], label="🔍 Transaction Analysis", tab_style={"margin": "5px"}, tabClassName="fw-bold"),

                        # Tab 4: Raw Data
                        dbc.Tab([
                            html.Div([
                                html.H5("Sample Transactions", className="mt-3 mb-3"),
                                dash_table.DataTable(
                                    id='raw-data-table',
                                    page_size=10,
                                    style_cell={
                                        'textAlign': 'left',
                                        'padding': '8px',
                                        'fontFamily': 'Arial, sans-serif'
                                    },
                                    style_header={
                                        'backgroundColor': '#6c757d',
                                        'color': 'white',
                                        'fontWeight': 'bold'
                                    }
                                )
                            ])
                        ], label="📄 Raw Data", tab_style={"margin": "5px"}, tabClassName="fw-bold")
                    ])
                ])
            ], className="shadow-sm")
        ])
    ])
], fluid=True, style={'padding': '20px', 'backgroundColor': '#f8f9fa', 'minHeight': '100vh'})

# Callback to update support value display
@app.callback(
    Output('support-value', 'children'),
    Input('support-slider', 'value')
)
def update_support_value(support):
    min_count = math.ceil(support * n_trans)
    return f"Current: {support:.3f} (≥ {min_count} transactions)"

# Main callback
@app.callback(
    [Output('results-table', 'data'),
     Output('results-table', 'columns'),
     Output('total-transactions', 'children'),
     Output('unique-items', 'children'),
     Output('min-support', 'children'),
     Output('itemsets-found', 'children'),
     Output('support-distribution', 'figure'),
     Output('itemset-length-dist', 'figure'),
     Output('top-itemsets-chart', 'figure'),
     Output('transaction-length-dist', 'figure'),
     Output('top-items-chart', 'figure'),
     Output('raw-data-table', 'data'),
     Output('raw-data-table', 'columns'),
     Output('results-info', 'children')],
    [Input('run-button', 'n_clicks')],
    [State('support-slider', 'value'),
     State('length-filter', 'value'),
     State('sort-by', 'value')]
)
def update_dashboard(n_clicks, min_support, length_filter, sort_by):
    if n_clicks == 0:
        # Initial state
        empty_fig = go.Figure()
        empty_fig.add_annotation(text="Click 'Run Apriori' to see results",
                               xref="paper", yref="paper", x=0.5, y=0.5,
                               showarrow=False, font_size=16)
        empty_fig.update_layout(paper_bgcolor='white', plot_bgcolor='white')

        # Sample raw data table
        sample_data = []
        for i, transaction in enumerate(transactions[:10]):
            sample_data.append({
                'Transaction ID': i + 1,
                'Items': ', '.join(transaction),
                'Item Count': len(transaction)
            })

        raw_columns = [{'name': col, 'id': col} for col in ['Transaction ID', 'Items', 'Item Count']]

        info_text = html.Div([
            html.P("Click the 'Run Apriori Algorithm' button above to start mining frequent itemsets.",
                  className="text-muted text-center")
        ])

        return ([], [], n_trans, len(items_list), "0.100", "0",
                empty_fig, empty_fig, empty_fig, empty_fig, empty_fig,
                sample_data, raw_columns, info_text)

    # Run Apriori algorithm
    min_count_required = math.ceil(min_support * n_trans)
    freq_itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)

    if not freq_itemsets.empty:
        # Compute accurate counts and additional info
        freq_itemsets['count'] = freq_itemsets['itemsets'].apply(lambda s: itemset_count(s, transactions))
        freq_itemsets['length'] = freq_itemsets['itemsets'].apply(lambda s: len(s))
        freq_itemsets['items'] = freq_itemsets['itemsets'].apply(lambda s: ', '.join(sorted(list(s))))

        # Apply length filter if specified
        if length_filter:
            freq_itemsets = freq_itemsets[freq_itemsets['length'] == length_filter]
            filter_text = f" (Filtered: {length_filter}-itemsets)"
        else:
            filter_text = ""

        # Sort results
        ascending = False if sort_by in ['support', 'count'] else True
        freq_itemsets = freq_itemsets.sort_values(sort_by, ascending=ascending)

        # Prepare table data
        display_df = freq_itemsets[['items', 'length', 'count', 'support']].copy()
        display_df['support'] = display_df['support'].round(4)
        table_data = display_df.rename(
            columns={'items': 'Itemset', 'length': 'Length', 'count': 'Count', 'support': 'Support'}
        ).to_dict('records')

        table_columns = [{'name': col, 'id': col} for col in ['Itemset', 'Length', 'Count', 'Support']]

        # Update statistics
        total_itemsets = len(freq_itemsets)

        # Info text
        info_text = html.Div([
            html.P(f"Found {total_itemsets} frequent itemsets with support \u2265 {min_support:.3f}{filter_text}",
                  className="text-success fw-bold")
        ])

        # Visualization 1: Support Distribution
        fig_support = px.histogram(freq_itemsets, x='support',
                                  title=f'Distribution of Support Values{filter_text}',
                                  labels={'support': 'Support', 'count': 'Number of Itemsets'},
                                  color_discrete_sequence=COLOR_PALETTES['support'])
        fig_support.update_layout(plot_bgcolor='white', paper_bgcolor='white')

        # Visualization 2: Itemset Length Distribution
        length_counts = freq_itemsets['length'].value_counts().sort_index()
        fig_length = px.pie(values=length_counts.values,
                           names=[f"{k}-Itemsets" for k in length_counts.index],
                           title=f'Itemset Length Distribution{filter_text}',
                           color_discrete_sequence=COLOR_PALETTES['length'])
        fig_length.update_traces(textposition='inside', textinfo='percent+label')

        # Visualization 3: Top Itemsets by Support
        top_20 = freq_itemsets.head(20)
        fig_top = px.bar(top_20, x='support', y='items', orientation='h',
                        title=f'Top 20 Itemsets by Support{filter_text}',
                        labels={'support': 'Support', 'items': 'Itemset'},
                        color='support',
                        color_continuous_scale='Viridis')
        fig_top.update_layout(yaxis={'categoryorder': 'total ascending'},
                            plot_bgcolor='white', paper_bgcolor='white')

        # Visualization 4: Transaction Length Distribution
        trans_lengths = [len(t) for t in transactions]
        fig_trans = px.histogram(x=trans_lengths,
                                title='Transaction Length Distribution',
                                labels={'x': 'Number of Items per Transaction', 'count': 'Frequency'},
                                color_discrete_sequence=['#4ECDC4'])
        fig_trans.update_layout(plot_bgcolor='white', paper_bgcolor='white')

        # Visualization 5: Top Individual Items with distinct colors
        item_counts = {}
        for transaction in transactions:
            for item in transaction:
                item_counts[item] = item_counts.get(item, 0) + 1

        top_items_df = pd.DataFrame({
            'item': list(item_counts.keys()),
            'count': list(item_counts.values())
        }).nlargest(15, 'count')

        # Assign distinct colors to each item
        fig_items = px.bar(top_items_df, x='count', y='item', orientation='h',
                          title='Top 15 Most Frequent Individual Items',
                          labels={'count': 'Frequency', 'item': 'Item'},
                          color='item',
                          color_discrete_sequence=COLOR_PALETTES['items'])
        fig_items.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                              showlegend=False)  # Hide legend for cleaner look

    else:
        # No itemsets found
        table_data = []
        table_columns = []
        total_itemsets = 0

        info_text = html.Div([
            html.P(f"No frequent itemsets found with support \u2265 {min_support:.3f}. Try lowering the support threshold.",
                  className="text-danger fw-bold")
        ])

        # Create empty figures with message
        empty_fig = go.Figure()
        empty_fig.add_annotation(text="No frequent itemsets found at this support level",
                               xref="paper", yref="paper", x=0.5, y=0.5,
                               showarrow=False, font_size=14)
        empty_fig.update_layout(paper_bgcolor='white', plot_bgcolor='white')

        fig_support = empty_fig
        fig_length = empty_fig
        fig_top = empty_fig

        # Transaction analysis figures still work
        trans_lengths = [len(t) for t in transactions]
        fig_trans = px.histogram(x=trans_lengths,
                                title='Transaction Length Distribution',
                                labels={'x': 'Number of Items per Transaction', 'count': 'Frequency'},
                                color_discrete_sequence=['#4ECDC4'])
        fig_trans.update_layout(plot_bgcolor='white', paper_bgcolor='white')

        item_counts = {}
        for transaction in transactions:
            for item in transaction:
                item_counts[item] = item_counts.get(item, 0) + 1

        top_items_df = pd.DataFrame({
            'item': list(item_counts.keys()),
            'count': list(item_counts.values())
        }).nlargest(15, 'count')

        # Assign distinct colors to each item
        fig_items = px.bar(top_items_df, x='count', y='item', orientation='h',
                          title='Top 15 Most Frequent Individual Items',
                          labels={'count': 'Frequency', 'item': 'Item'},
                          color='item',
                          color_discrete_sequence=COLOR_PALETTES['items'])
        fig_items.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                              showlegend=False)

    # Sample raw data table
    sample_data = []
    for i, transaction in enumerate(transactions[:10]):
        sample_data.append({
            'Transaction ID': i + 1,
            'Items': ', '.join(transaction),
            'Item Count': len(transaction)
        })

    raw_columns = [{'name': col, 'id': col} for col in ['Transaction ID', 'Items', 'Item Count']]

    return (table_data, table_columns,
            n_trans, len(items_list), f"{min_support:.3f}", total_itemsets,
            fig_support, fig_length, fig_top, fig_trans, fig_items,
            sample_data, raw_columns, info_text)

# ---------------------------
# Ngrok Setup and Server Start
# ---------------------------
print("🔧 Setting up ngrok tunnel...")

# Kill any existing tunnels
ngrok.kill()

# Set your ngrok authtoken. Replace 'YOUR_AUTHTOKEN' with your actual token.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("359zqcVb5UeF5nUBm3ubqpUUFmY_4c2a8pLE3m76uJ4mJcHQk")

# Create a tunnel - using port 8051 (changed from 8050)
public_url = ngrok.connect(8051, bind_tls=True)
print(f"✅ Ngrok tunnel created successfully!")
print(f"🌐 Your dashboard is now available at: {public_url}")
print("📱 You can share this URL with anyone to access your dashboard")
print("⏳ Starting Dash server...")

# Run the server
if __name__ == '__main__':
    app.run(
        host='0.0.0.0',
        port=8051, # Changed from 8050
        debug=False,  # Set to False for better ngrok performance
        use_reloader=False  # Disable reloader for ngrok
    )

Loading data...
✅ Loaded 15 transactions
🔧 Setting up ngrok tunnel...
✅ Ngrok tunnel created successfully!
🌐 Your dashboard is now available at: NgrokTunnel: "https://sharika-uncordial-camie.ngrok-free.dev" -> "http://localhost:8051"
📱 You can share this URL with anyone to access your dashboard
⏳ Starting Dash server...


<IPython.core.display.Javascript object>